In [5]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 26.5 MB/s eta 0:00:00


In [1]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
INPUT_PATH = "/content/drive/MyDrive/dataset_raw_candidates.jsonl"
OUTPUT_PATH = "/content/drive/MyDrive/dataset_final.jsonl"
FINAL_SIZE = 200

STYLE_INSTRUCTION = (
    "Перепиши следующий ответ в стиле: кратко, по делу, без вводных фраз и максимум 3-4 предложения"
    "Сохрани смысл, не добавляй новых фактов.\n\nОтвет для переписывания:\n{answer}"
)

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [4]:
def restyle(answer):
    prompt = f"[INST] {STYLE_INSTRUCTION.format(answer=answer)} [/INST]"
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    text = tok.decode(out[0], skip_special_tokens=True)
    return text.split("[/INST]")[-1].strip()

In [5]:
test_answer = "The three primary colors are red, blue, and yellow."
print(restyle(test_answer))

Red, blue, and yellow are the three primary colors.


In [6]:
result = restyle(test_answer)
print( (result))

Red, blue, and yellow are the three primary colors.


In [11]:
with open(INPUT_PATH, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

print(f"{len(rows)}")

250


In [12]:
results = []

for i, row in enumerate(rows[:FINAL_SIZE]):
    new_resp = restyle(row["response"])

    if len(new_resp) < 10 or new_resp == row["response"]:
        continue

    results.append({"instruction": row["instruction"], "response": new_resp})

    if i % 20 == 0:
        print(f"Обработано {i}/{FINAL_SIZE}")

print(f"\n {len(results)} ")

Обработано 0/200
Обработано 20/200
Обработано 40/200
Обработано 60/200
Обработано 80/200
Обработано 100/200
Обработано 120/200
Обработано 140/200
Обработано 160/200
Обработано 180/200

 200 


In [15]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"{len(results)}, {OUTPUT_PATH}")

200, /content/drive/MyDrive/dataset_final.jsonl


In [17]:
!head -5 /content/drive/MyDrive/dataset_final.jsonl

{"instruction": "Give three tips for staying healthy.", "response": "Maintain a balanced diet with ample fruits and vegetables. Exercise regularly. Keep sleep consistent and sufficient."}
{"instruction": "What are the three primary colors?", "response": "Red, blue, and yellow are the three primary colors."}
{"instruction": "Describe the structure of an atom.", "response": "Atom consists of a charged nucleus, containing protons and neutrons, surrounded by orbiting electrons. Protons and neutrons positively charged, electrons negatively charged. Total charge neutral. Nucleus composition defines atomic number, atom type."}
{"instruction": "How can we reduce air pollution?", "response": "1. Reduce air pollution through renewable energy.\n2. Encourage public transportation use and ban fossil fuel burning.\n3. Implement emissions policies for industries and vehicles.\n4. Individuals: Reduce car use, avoid wood burning, and use energy-efficient appliances."}
{"instruction": "Describe a time w

In [18]:
import json

In [20]:
with open('/content/drive/MyDrive/dataset_final.jsonl', encoding='utf-8') as f:
    rows = [json.loads(line) for line in f]

for r in rows[50:65]:  # середина файла
    print("Q:", r["instruction"])
    print("A:", r["response"])
    print("\n")

Q: Find the synonyms of the following word: 'Tenacious'.
A: Determined, unyielding, steadfast, persistent.


Q: Generate a creative birthday wish for a friend.
A: Wish you a joyful birthday. May it bring happiness, laughter, and blessings. Enjoy a wonderful year ahead.


Q: Compose a five word sentence describing your day.
A: Day exceptional, filled with wonder.


Q: Search the web and find the title of the longest novel.
A: Longest novel title: "Gordale" by Carolyn Redfearn.


Q: Compile a list of 5 US states located in the Mid West.
A: Midwest: Illinois, Indiana, Michigan, Ohio, Wisconsin.


Q: During the last member meeting, create a list of 5 ideas to improve the club.
A: Last meeting: Increase meeting frequency. Propose mentorship program. Create club website. Establish activity budget. Offer participation incentives.


Q: Write a simple definition of the word "economics".
A: Economics is the study of resource allocation in production and distribution of goods and services.


Q: W